# Module 5, Section 2: Eval-Driven Development

The benefit of building out a dataset and eval metrics upfront is that we have a quantitative signal that we can use to understand how changes to our app impact downstream performance. Without this, we're just flying blind!


<div align="center">
    <img src="../../images/eval_process.png">
</div>

### Systematically Improving Our Application

Our baseline evaluation revealed weaknesses in the deep agent's order specialist:

**The Problem:**
- ❌ **Excessive tool calls**: Sequential chains of `get_customer_orders` → multiple `get_order_items` → multiple `get_order_item_price` calls, etc.
- ❌ **Missing capabilities**: Can't aggregate (SUM, COUNT, AVG), filter (WHERE), or JOIN across tables - so relies on in-memory math
- ❌ **Limited flexibility**: Rigid tools can't adapt to complex or novel queries, and none of them reads the customer's own record

**Possible Solution:**
Give the order specialist a flexible SQL tool that generates custom queries at runtime, scoped to the signed-in customer's data.

Let's use our evaluation setup to drive a targeted improvement!

## 1. Setup

We'll give the order specialist a SQL tool that can answer complex questions about the customer's orders with a single query.

<div align="center">
    <img src="../../images/order_specialist_improvement.svg" width="700">
</div>

In [ ]:
from dotenv import load_dotenv
from tools.database import query_my_orders

load_dotenv()

# The tool's description carries the schema the model needs to write queries
print(query_my_orders.description)

## 2. Build SQL Agent

In Module 2 the SQL agent got `execute_sql`, a tool that runs any SELECT. That can't work here: the order specialist serves a signed-in customer, and a raw SQL tool would let them read **every** customer's orders. The customer-scope guard only checks `customer_id` and `order_id` arguments, and a SQL string has neither.

So the tool enforces the scope itself, from the verified customer in run context:

```
SupportContext(customer_id="CUST-010")      set by the app, not the conversation
        │
        ▼
query_my_orders(sql)
   1. read-only connection to techhub.db
   2. temp views: orders, order_items, customers  ← only CUST-010's rows
   3. authorizer: reads of the views and the public products table only
   4. run the query (row cap, 2-second limit)
        │
        ▼
"SELECT ... FROM orders"  sees CUST-010's orders, and nobody else's
```

In [ ]:
import inspect

print(inspect.getsource(query_my_orders.func))

Now an agent that writes those queries, using the `create_agent()` abstraction with the same context schema as the deep agent:

In [ ]:
from langchain.agents import create_agent
from agents.deep_support_agent import SupportContext
from config import DEFAULT_MODEL

sql_agent = create_agent(
    model=DEFAULT_MODEL,
    tools=[query_my_orders],
    name="sql_agent",
    system_prompt="You are TechHub's order specialist. Answer with one SQL query when you can, and report exactly what it returns.",
    context_schema=SupportContext,
)
sql_agent

#### 🔒 Production Security Note

In Module 2, the SQL tool blocked writes with keyword filtering. This tool also has to keep each customer inside their own data, so it uses several layers:

| Layer | What it blocks |
|---|---|
| Read-only connection (`mode=ro`) | Any write to the database |
| Temp views over the verified customer's rows | Reading another customer's orders, order items, or account |
| SQLite authorizer | Reading the underlying tables directly, `PRAGMA`, `ATTACH`, writes |
| Row cap and 2-second limit | Huge results and runaway queries |
| Customer ID from run context, validated | The conversation choosing whose data to read |

In production PostgreSQL, you'd add database-level controls too: a read-only role and row-level security keyed to the signed-in customer.

## 3. Quick Demo

Let's test the SQL agent on a query that previously took a chain of lookups, as the customer who owns the order:

In [ ]:
# Test on a query that requires joining orders, items, and products
question = "What items were in order ORD-2023-0002? How much did each cost?"

result = sql_agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=SupportContext(customer_id="CUST-010"),
)

for message in result["messages"]:
    message.pretty_print()

✅ **Success!** The SQL agent generates a custom query with JOINs to answer the question in a single tool call.

Now the same question from a **different** customer. The query runs, but only over that customer's rows, so the other customer's order isn't there:

In [ ]:
result = sql_agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=SupportContext(customer_id="CUST-003"),
)

result["messages"][-1].pretty_print()

Check the LangSmith traces to see the generated SQL queries!

## 4. Integrate SQL Agent with the Deep Agent

Now let's swap the order specialist's rigid tools for this SQL tool in the full deep agent. `order_tools="sql"` gives the order specialist `query_my_orders` instead of the five lookups:

<div align="center">
    <img src="../../images/order_specialist_improvement.svg" width="500">
</div>

In [ ]:
from agents.deep_support_agent import create_deep_support_agent

# Compose the deep agent with the SQL order specialist instead of the rigid one
improved_agent = create_deep_support_agent(order_tools="sql")

Quick test:

In [ ]:
import uuid

# New thread
thread_id = uuid.uuid4()
config = {"configurable": {"thread_id": thread_id}}

# The app has signed the customer in, so their ID comes from run context
result = improved_agent.invoke(
    {"messages": [{"role": "user", "content": "Whats the status of my recent order?"}]},
    config=config,
    context=SupportContext(customer_id="CUST-003"),
)

In [ ]:
for message in result["messages"]:
    message.pretty_print()

## 5. Re-evaluate Performance

Now let's run the same evaluation on the same dataset to quantitatively see how it impacts our performance metrics.

In [ ]:
# Import evaluators from Section 1 (now packaged in evaluators module)
from evaluators import (
    correctness_evaluator,
    count_total_tool_calls_evaluator,
    order_specialist_tool_calls_evaluator,
)

In [ ]:
from langsmith import Client

client = Client()

AGENT = improved_agent

import re
import sqlite3
import uuid
from pathlib import Path

from config import DEFAULT_DB_PATH

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")


def customer_id_for(question: str) -> str | None:
    """Look up the customer the way sign-in would: by the email they gave."""
    match = EMAIL.search(question)
    if not match:
        return None
    with sqlite3.connect(f"{Path(DEFAULT_DB_PATH).resolve().as_uri()}?mode=ro", uri=True) as conn:
        row = conn.execute("SELECT customer_id FROM customers WHERE email = ?", (match.group(0),)).fetchone()
    return row[0] if row else None


def target_function(inputs: dict) -> dict:
    """Target function that runs our agent to get outputs for evaluation."""

    thread_id = uuid.uuid4()
    config = {"configurable": {"thread_id": thread_id}}
    question = inputs["messages"][-1]["content"]

    result = AGENT.invoke(
        inputs,
        config=config,
        context=SupportContext(customer_id=customer_id_for(question)),
    )

    return {
        "messages": [{"role": "assistant", "content": result["messages"][-1].text}]
    }


# The Section 1 dataset: the most recent one with its name prefix
dataset = max(
    client.list_datasets(dataset_name_contains="techhub-deep-agent-baseline-"),
    key=lambda ds: ds.created_at,
)

# Run evaluation on same dataset
results = client.evaluate(
    target_function,
    data=dataset.name,  # Same dataset as section 1
    evaluators=[correctness_evaluator, count_total_tool_calls_evaluator, order_specialist_tool_calls_evaluator],
    experiment_prefix="deep-agent-sql",
    description="Evaluate the SQL order specialist's flexible queries vs baseline rigid tools",
    max_concurrency=5,
)

## 6. Compare Results in LangSmith

Now let's analyze our improvement using LangSmith's comparison features. Open both experiments side by side:

In [ ]:
experiments = sorted(client.list_projects(reference_dataset_id=dataset.id), key=lambda p: p.start_time)
baseline = next(p for p in reversed(experiments) if p.name.startswith("deep-agent-baseline"))
improved = next(p for p in reversed(experiments) if p.name.startswith("deep-agent-sql"))

print("Baseline:", baseline.name)
print("Improved:", improved.name)
print(f"📊 Compare: {dataset.url}/compare?selectedSessions={baseline.id},{improved.id}")

What to expect:

| Metric | Direction | Why |
|---|---|---|
| `order_specialist_tool_calls` | ⬇️ Down sharply | One query replaces a chain of lookups |
| `total_tool_calls` | ⬇️ Down | Same reason; delegations and doc searches stay about the same |
| `correctness` | ⬆️ Same or better | Aggregations are computed by SQL instead of in the model's head, and the customer record is now readable |
| Cross-customer example | Still refused | The SQL only ever sees the signed-in customer's rows |

Some answers will still be marked wrong for adding detail or masking data for privacy. That's the judge, not the agent: aligning the judge is next.